# 02 — Virtual Environments and Project Dependencies

Every Python project you work on will eventually need packages. The problem is that different projects often need **different versions** of the same package — and Python can only have one version of a package installed at a time.

**Virtual environments** solve this by giving each project its own isolated copy of Python and its own package directory.

By the end of this module you'll be able to:
- Create, activate, and deactivate virtual environments
- Understand what isolation means and why it matters
- Choose the right dependency declaration strategy for applications vs. libraries

---

## 1. The problem virtual environments solve

Imagine you have two projects:
- **Project A** is a legacy app that uses `flask==2.2`
- **Project B** is a new API that requires `flask>=3.0`

Without isolation, only one version of Flask can be installed globally at a time. Installing for Project B breaks Project A.

In [1]:
# Simulate the conflict scenario

projects = {
    "project-a": {"flask": "2.2.5", "werkzeug": "2.3.0"},
    "project-b": {"flask": "3.0.3", "werkzeug": "3.0.3"},
}

print("Without virtual environments:")
print("  You can only have ONE version of each package globally.")
print()

for project, deps in projects.items():
    print(f"  {project} needs:")
    for pkg, ver in deps.items():
        print(f"    {pkg}=={ver}")
    print()

print("Result: installing either project's deps breaks the other.")
print()
print("With virtual environments:")
print("  Each project gets its own isolated directory.")
print("  Both can coexist with no conflict.")

Without virtual environments:
  You can only have ONE version of each package globally.

  project-a needs:
    flask==2.2.5
    werkzeug==2.3.0

  project-b needs:
    flask==3.0.3
    werkzeug==3.0.3

Result: installing either project's deps breaks the other.

With virtual environments:
  Each project gets its own isolated directory.
  Both can coexist with no conflict.


---

## 2. Creating a virtual environment

Python ships `venv` in the standard library — no installation needed.

```bash
# Create a virtual environment in a folder named .venv
python -m venv .venv
```

> **Convention:** `.venv` (with a leading dot) is the widely accepted name. It stays hidden in most file explorers and is auto-detected by VS Code, PyCharm, and other editors.

In [2]:
import subprocess, tempfile, os, sys
from pathlib import Path

# Create a temporary venv to inspect its structure
tmpdir = tempfile.mkdtemp()
venv_path = Path(tmpdir) / ".venv"

print("Creating virtual environment...")
result = subprocess.run(
    [sys.executable, "-m", "venv", str(venv_path)],
    capture_output=True, text=True
)

if result.returncode == 0:
    print(f"Created at: {venv_path}")
    print()
    print("Top-level contents:")
    for item in sorted(venv_path.iterdir()):
        print(f"  {item.name}/" if item.is_dir() else f"  {item.name}")
else:
    print("Error:", result.stderr)

Creating virtual environment...
Created at: /var/folders/l3/6dn78ndx5cb9nbc_4r6nycq40000gn/T/tmpjh7jp_zx/.venv

Top-level contents:
  .gitignore
  bin/
  include/
  lib/
  pyvenv.cfg


### What's inside a virtual environment?

```
.venv/
├── bin/                    ← executables (Scripts\ on Windows)
│   ├── python              ← symlink to the interpreter
│   ├── pip
│   └── activate            ← the script you source to activate
├── include/
├── lib/
│   └── python3.x/
│       └── site-packages/  ← installed packages land here
└── pyvenv.cfg              ← points back to the base interpreter
```

Activating a virtual environment prepends `.venv/bin` to your shell's `PATH`. From that point, `python` and `pip` resolve to the virtual environment's copies.

In [3]:
# Inspect the structure of our temp venv

# Read pyvenv.cfg — tells you where this venv came from
pyvenv_cfg = venv_path / "pyvenv.cfg"
print("pyvenv.cfg:")
print(pyvenv_cfg.read_text())

# List the bin/ directory
bin_dir = venv_path / "bin" if (venv_path / "bin").exists() else venv_path / "Scripts"
print(f"{bin_dir.name}/ contents:")
for f in sorted(bin_dir.iterdir())[:12]:
    print(f"  {f.name}")

pyvenv.cfg:
home = /opt/homebrew/Cellar/python@3.14/3.14.6/Frameworks/Python.framework/Versions/3.14/bin
include-system-site-packages = false
version = 3.14.6
executable = /opt/homebrew/Cellar/python@3.14/3.14.6/Frameworks/Python.framework/Versions/3.14/bin/python3.14
command = /opt/homebrew/Cellar/jupyterlab/4.6.2/libexec/bin/python -m venv /var/folders/l3/6dn78ndx5cb9nbc_4r6nycq40000gn/T/tmpjh7jp_zx/.venv

bin/ contents:
  Activate.ps1
  activate
  activate.csh
  activate.fish
  pip
  pip3
  pip3.14
  python
  python3
  python3.14
  𝜋thon


In [4]:
# Fresh venv has almost nothing in site-packages
site_packages = list((venv_path / "lib").glob("*/site-packages"))
if site_packages:
    sp = site_packages[0]
    packages = [d.name for d in sp.iterdir() if not d.name.startswith("_")]
    print(f"site-packages ({sp}):")
    print(f"  {len(packages)} items (just pip, setuptools, etc.)")
    for p in sorted(packages)[:10]:
        print(f"  - {p}")
else:
    print("site-packages not found")

site-packages (/var/folders/l3/6dn78ndx5cb9nbc_4r6nycq40000gn/T/tmpjh7jp_zx/.venv/lib/python3.14/site-packages):
  2 items (just pip, setuptools, etc.)
  - pip
  - pip-26.1.2.dist-info


---

## 3. Activating and deactivating

Activation updates your shell's `PATH` so the venv's Python and pip are used by default.

```bash
# macOS / Linux
source .venv/bin/activate

# Windows (PowerShell)
.venv\Scripts\Activate.ps1

# Windows (cmd.exe)
.venv\Scripts\activate.bat
```

Your prompt changes to show `(.venv)` as a reminder that you're inside the environment.

```bash
# When you're done:
deactivate
```

In [5]:
# How Python detects whether it's inside a virtual environment
import sys

def venv_status():
    in_venv = hasattr(sys, "real_prefix") or (
        hasattr(sys, "base_prefix") and sys.base_prefix != sys.prefix
    )
    return {
        "in_venv": in_venv,
        "executable": sys.executable,
        "prefix": sys.prefix,
        "base_prefix": getattr(sys, "base_prefix", sys.prefix),
    }

status = venv_status()
print(f"Inside a venv : {status['in_venv']}")
print(f"Executable    : {status['executable']}")
print(f"sys.prefix    : {status['prefix']}")
print(f"base_prefix   : {status['base_prefix']}")
print()
if status["in_venv"]:
    print("pip install will install into THIS virtual environment only.")
else:
    print("pip install would install into the global environment — activate a venv first!")

Inside a venv : True
Executable    : /opt/homebrew/Cellar/jupyterlab/4.6.2/libexec/bin/python
sys.prefix    : /opt/homebrew/Cellar/jupyterlab/4.6.2/libexec
base_prefix   : /opt/homebrew/opt/python@3.14/Frameworks/Python.framework/Versions/3.14

pip install will install into THIS virtual environment only.


### Demonstrating isolation

Each virtual environment starts empty. Installing into one does not affect another.

In [6]:
# Create two separate venvs and install different packages into each
import subprocess, tempfile, sys
from pathlib import Path

base = Path(tempfile.mkdtemp())
venv_a = base / "env_a"
venv_b = base / "env_b"

for venv in [venv_a, venv_b]:
    subprocess.run([sys.executable, "-m", "venv", str(venv)], capture_output=True)

# Python executable inside each venv
def pip_in(venv: Path):
    bin_dir = venv / "bin" if (venv / "bin").exists() else venv / "Scripts"
    return str(bin_dir / "pip")

# Install different packages into each
subprocess.run([pip_in(venv_a), "install", "httpx==0.27.0", "-q"], capture_output=True)
subprocess.run([pip_in(venv_b), "install", "httpx==0.26.0", "-q"], capture_output=True)

# Show what's in each
for name, venv in [("env_a", venv_a), ("env_b", venv_b)]:
    result = subprocess.run([pip_in(venv), "show", "httpx"], capture_output=True, text=True)
    version = next(
        (l.split(":")[1].strip() for l in result.stdout.splitlines() if l.startswith("Version")),
        "not installed"
    )
    print(f"{name}: httpx {version}")

print()
print("Both versions coexist — each environment is completely isolated.")

env_a: httpx 0.27.0
env_b: httpx 0.26.0

Both versions coexist — each environment is completely isolated.


---

## 4. Dependency strategies: Applications vs. Libraries

How you specify dependencies depends on what you're building. This is one of the most important distinctions in Python packaging.

### Applications

An application is something **you deploy and run yourself** — a Django site, a FastAPI service, a data pipeline, a CLI tool for your team.

**Rule: pin exact versions.**

You want every deployment — dev, staging, production — to use identical packages. A surprise upgrade on a Friday evening is not acceptable.

```text
# requirements.txt — an application
fastapi==0.111.0
uvicorn==0.30.1
sqlalchemy==2.0.31
pydantic==2.7.4
alembic==1.13.2
python-dotenv==1.0.1
```

### Libraries

A library is something **other developers install** as a dependency of their own projects. You publish it to PyPI.

**Rule: use version ranges (don't pin).**

If your library pins `httpx==0.27.0`, you force that exact version on every user. If their app needs `httpx==0.26.0` for another reason, they can't install both — your library makes their project impossible to resolve.

```toml
# pyproject.toml — a library
[project]
dependencies = [
    "httpx>=0.25",        # any 0.25 or newer works
    "pydantic>=2.0,<3",   # 2.x series only — we know 3.0 breaks us
]
```

In [7]:
from typing import NamedTuple

class Dep(NamedTuple):
    package: str
    app_pin: str     # what an application would write
    lib_range: str   # what a library would write
    reason: str

deps = [
    Dep("fastapi", "fastapi==0.111.0", "fastapi>=0.100",      "breaking changes between minors"),
    Dep("sqlalchemy", "sqlalchemy==2.0.31", "sqlalchemy>=2.0,<3", "major version rewrite in 2.0"),
    Dep("pydantic", "pydantic==2.7.4", "pydantic>=2.0",      "2.x API is stable"),
    Dep("httpx", "httpx==0.27.0", "httpx>=0.25",         "backward-compatible patches"),
]

print(f"{'Package':<12} {'App (pinned)':<22} {'Library (range)':<22} {'Reason'}")
print("-" * 85)
for d in deps:
    print(f"{d.package:<12} {d.app_pin:<22} {d.lib_range:<22} {d.reason}")

Package      App (pinned)           Library (range)        Reason
-------------------------------------------------------------------------------------
fastapi      fastapi==0.111.0       fastapi>=0.100         breaking changes between minors
sqlalchemy   sqlalchemy==2.0.31     sqlalchemy>=2.0,<3     major version rewrite in 2.0
pydantic     pydantic==2.7.4        pydantic>=2.0          2.x API is stable
httpx        httpx==0.27.0          httpx>=0.25            backward-compatible patches


In [8]:
# Python's packaging library can evaluate version constraints
from importlib.metadata import version
try:
    from packaging.version import Version
    from packaging.specifiers import SpecifierSet

    examples = [
        ("httpx", ">=0.25,<1.0"),
        ("pydantic", ">=2.0,<3"),
        ("rich", ">=13.0"),
    ]

    print("Checking installed versions against library-style ranges:")
    print()
    for pkg, spec_str in examples:
        try:
            installed = version(pkg)
            spec = SpecifierSet(spec_str)
            ok = Version(installed) in spec
            status = "PASS" if ok else "FAIL"
            print(f"  {pkg} {installed} satisfies {spec_str!r}: {status}")
        except Exception:
            print(f"  {pkg}: not installed")

except ImportError:
    print("packaging library not installed — run: pip install packaging")

Checking installed versions against library-style ranges:

  httpx 0.28.1 satisfies '>=0.25,<1.0': PASS
  pydantic 2.13.5 satisfies '>=2.0,<3': PASS
  rich 15.0.0 satisfies '>=13.0': PASS


---

## 5. End-to-end workflow

Here's the complete flow from starting a project to sharing it with a teammate.

In [9]:
workflow = [
    ("Create project dir",      "mkdir my-app && cd my-app"),
    ("Create venv",             "python -m venv .venv"),
    ("Activate (macOS/Linux)",  "source .venv/bin/activate"),
    ("Activate (Windows)",      ".venv\\Scripts\\activate"),
    ("Install deps",            "pip install fastapi uvicorn"),
    ("Pin deps",                "pip freeze > requirements.txt"),
    ("Commit",                  "git add requirements.txt && git commit"),
    ("--- teammate clones ---",  ""),
    ("Restore",                 "python -m venv .venv"),
    ("Activate",                "source .venv/bin/activate"),
    ("Install from lock",       "pip install -r requirements.txt"),
]

print(f"{'Step':<28} {'Command'}")
print("-" * 65)
for step, cmd in workflow:
    print(f"{step:<28} {cmd}")

Step                         Command
-----------------------------------------------------------------
Create project dir           mkdir my-app && cd my-app
Create venv                  python -m venv .venv
Activate (macOS/Linux)       source .venv/bin/activate
Activate (Windows)           .venv\Scripts\activate
Install deps                 pip install fastapi uvicorn
Pin deps                     pip freeze > requirements.txt
Commit                       git add requirements.txt && git commit
--- teammate clones ---      
Restore                      python -m venv .venv
Activate                     source .venv/bin/activate
Install from lock            pip install -r requirements.txt


### One venv per project — always

Never share a virtual environment between projects. The cost of creating a new one is seconds. The cost of debugging a dependency conflict is hours.

In [10]:
# Common virtual environment naming conventions
conventions = [
    (".venv",    "Most common — hidden, auto-detected by VS Code/PyCharm"),
    ("venv",     "Also common — visible in file explorers"),
    ("env",      "Older convention — can clash with .env config files"),
    (".env",     "Avoid! Conflicts with dotenv files (.env for secrets)"),
]

print(f"{'Name':<10} {'Notes'}")
print("-" * 60)
for name, note in conventions:
    recommended = " ← recommended" if name == ".venv" else (" ← avoid" if name == ".env" else "")
    print(f"{name:<10} {note}{recommended}")

Name       Notes
------------------------------------------------------------
.venv      Most common — hidden, auto-detected by VS Code/PyCharm ← recommended
venv       Also common — visible in file explorers
env        Older convention — can clash with .env config files
.env       Avoid! Conflicts with dotenv files (.env for secrets) ← avoid


In [11]:
# Clean up temp directories created during this notebook
import shutil
for d in [venv_path.parent, venv_a.parent, venv_b.parent, base]:
    try:
        shutil.rmtree(d, ignore_errors=True)
    except Exception:
        pass
print("Temporary venvs cleaned up.")

Temporary venvs cleaned up.


---

## Summary

| Concept | Key point |
|---|---|
| `python -m venv .venv` | Create an isolated environment |
| `source .venv/bin/activate` | Activate it (macOS/Linux) |
| `deactivate` | Return to the global environment |
| Isolation | Each venv has its own `site-packages`; installs don't affect other envs |
| Applications | Pin exact versions (`pkg==x.y.z`) |
| Libraries | Use ranges (`pkg>=x.y`) so you don't constrain users |

**The golden rule:** one virtual environment per project, always.

---

**Next:** [03 — Python Environments and Version Control](../03-python-environments-and-version-control/python-environments-and-version-control.ipynb)

---

## Exercises

1. **Prove isolation** — create two virtual environments. Install `rich==13.0.0` in the first and `rich==13.7.0` in the second. Use `pip show rich` inside each to confirm different versions coexist.

2. **Inspect a fresh venv** — create a new `.venv`, activate it, and run `pip list`. How many packages are pre-installed? What are they?

3. **Application vs. library** — you're building two things:
   - A web scraper you'll deploy on a cron job
   - A reusable HTML-parsing library you'll publish to PyPI
   
   For each, decide whether to pin exact versions or use ranges for `requests` and `beautifulsoup4`. Justify your choice.

4. **Simulate `sys.prefix` vs `sys.base_prefix`** — with a venv active, print both values. Then deactivate and print them again. What changes? Why does the difference tell Python it's inside a venv?